# Phase 6 v3.1 — Evaluation (for v3.1 Fine-Tuned Model)

**v2 → v3.1 model changes:**
1. Response-only loss masking (100% of gradient on prediction values)
2. REMOVED modules_to_save (no tied weight corruption)
3. 3 epochs + early stopping (was 1 epoch)
4. LR 1e-4 (was 2e-4 — preserves pretrained features)
5. MSI-H oversampling DISABLED (4 patients = memorization, not learning)

**Evaluation changes from v2:**
- Points to `models/immunopath-v3.1` adapters
- Results to `results/phase6_v3.1`
- Updated comparison report with v3.1 fixes

---
**Targets:**
| Task | Target |
|------|--------|
| CD274 AUC | > 0.70 |
| MSI AUC | > 0.75 |
| TME Accuracy | > 0.65 |
| TIL Spearman ρ | > 0.60 |
| Immune Score MAE | < 0.15 |
| ECE | < 0.10 |

In [6]:
import os, subprocess

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/ImmunoPath"
DATA_DIR = f"{PROJECT_DIR}/data"
RESULTS_DIR = f"{PROJECT_DIR}/results/phase6_v3.1"
PHASE4_DIR = f"{PROJECT_DIR}/results/phase4"

# --- V3 model paths ---
TRAINING_DIR = f"{DATA_DIR}/training_v3"  # v3.1 training data splitting data
MODEL_DIR = f"{PROJECT_DIR}/models/immunopath-v3.1"
ADAPTER_DIR = f"{MODEL_DIR}/lora_adapters"

# Fallback chain: v3.1 → v2 → v1
if not os.path.exists(ADAPTER_DIR):
    if os.path.exists(f"{MODEL_DIR}/adapter_config.json"):
        ADAPTER_DIR = MODEL_DIR
        print(f"Using adapters from MODEL_DIR root: {ADAPTER_DIR}")
    else:
        print(f"v3.1 adapters not found at {ADAPTER_DIR}")
        # Try v2 as fallback
        ADAPTER_DIR_V2 = f"{PROJECT_DIR}/models/immunopath-v2/lora_adapters"
        if os.path.exists(ADAPTER_DIR_V2):
            ADAPTER_DIR = ADAPTER_DIR_V2
            print(f"  Falling back to v2 adapters: {ADAPTER_DIR}")

# Fallback for test data
if not os.path.exists(f"{TRAINING_DIR}/test.jsonl"):
    print("training_v3 not found, falling back to training/")
    TRAINING_DIR = f"{DATA_DIR}/training"

MODEL_ID = "google/medgemma-1.5-4b-it"

os.makedirs(RESULTS_DIR, exist_ok=True)

subprocess.run([
    "pip", "install", "-q", "--no-cache-dir",
    "transformers>=4.50.0",
    "accelerate>=0.34.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.44.0",
    "pandas", "numpy", "scikit-learn", "scipy",
    "matplotlib", "seaborn", "tqdm",
], check=True)


FLASH_ATTN_AVAILABLE = False

from huggingface_hub import login
from google.colab import userdata
try:
    login(token=userdata.get('HF_TOKEN'))
    print("HuggingFace login ✓")
except Exception:
    print("HF_TOKEN not set")

import torch, gc
import numpy as np
import pandas as pd
import json, time
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

print(f"\nPaths:")
print(f"  TRAINING_DIR: {TRAINING_DIR}")
print(f"  MODEL_DIR:    {MODEL_DIR}")
print(f"  ADAPTER_DIR:  {ADAPTER_DIR}")
print(f"  RESULTS_DIR:  {RESULTS_DIR}")
print(f"  MODEL_ID:     {MODEL_ID}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HuggingFace login ✓
GPU: NVIDIA L4 (23.7 GB)

Paths:
  TRAINING_DIR: /content/drive/MyDrive/ImmunoPath/data/training_v3
  MODEL_DIR:    /content/drive/MyDrive/ImmunoPath/models/immunopath-v3.1
  ADAPTER_DIR:  /content/drive/MyDrive/ImmunoPath/models/immunopath-v3.1/lora_adapters
  RESULTS_DIR:  /content/drive/MyDrive/ImmunoPath/results/phase6_v3.1
  MODEL_ID:     google/medgemma-1.5-4b-it


## Load Fine-Tuned Model (Base + v3.1 LoRA Adapters)

v3.1 has NO `modules_to_save`, so PeftModel loading is clean — no weight tying warnings.

In [7]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

attn_impl = "flash_attention_2" if FLASH_ATTN_AVAILABLE else "eager"

print(f"Loading base model: {MODEL_ID} (4-bit quantized)")
processor = AutoProcessor.from_pretrained(MODEL_ID)

# Match training quantization config exactly
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation=attn_impl,
    quantization_config=bnb_config,
)

print(f"Loading v3.1 LoRA adapters from {ADAPTER_DIR}")
print(f"  (v3.1: no modules_to_save — weight tying preserved)")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

# LEFT padding for inference
processor.tokenizer.padding_side = "left"

allocated = torch.cuda.memory_allocated() / 1e9
print(f"Fine-tuned model loaded ({attn_impl}), VRAM: {allocated:.2f} GB")
print(f"Padding side: {processor.tokenizer.padding_side}")
print(f"tie_word_embeddings: {base_model.config.tie_word_embeddings}")

Loading base model: google/medgemma-1.5-4b-it (4-bit quantized)


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Loading v3.1 LoRA adapters from /content/drive/MyDrive/ImmunoPath/models/immunopath-v3.1/lora_adapters
  (v3.1: no modules_to_save — weight tying preserved)
Fine-tuned model loaded (eager), VRAM: 3.40 GB
Padding side: left
tie_word_embeddings: True


## Load Test Data + Ground Truth

In [8]:
def load_jsonl(path: str) -> list:
    samples = []
    with open(path) as f:
        for line in f:
            if line.strip():
                samples.append(json.loads(line.strip()))
    return samples


def normalize_prediction_keys(parsed_json: dict) -> dict:
    """Map variant keys to canonical schema keys."""
    if not parsed_json:
        return parsed_json
    KEY_MAP = {
        "cd274_rna_proxy_level": "cd274_expression",
        "cd274_rna_proxy": "cd274_expression",
        "cd274_proxy": "cd274_expression",
        "cd274_proxy_level": "cd274_expression",
        "cd274_level": "cd274_expression",
        "pdl1_expression": "cd274_expression",
        "overall_immune_score": "immune_score",
        "cd8_t_cell_infiltration": "cd8_infiltration",
        "tme_subtype_confidence": "prediction_entropy",
        "til_bucket": "til_density",
        "msi_probability": "msi_probability",
        "uncertainty": "prediction_entropy",
    }
    normalized = {}
    for k, v in parsed_json.items():
        k_lower = k.lower()
        normalized[KEY_MAP.get(k_lower, k_lower)] = v
    return normalized


test_samples = load_jsonl(f"{TRAINING_DIR}/test.jsonl") if os.path.exists(f"{TRAINING_DIR}/test.jsonl") else []
val_samples = load_jsonl(f"{TRAINING_DIR}/val.jsonl") if os.path.exists(f"{TRAINING_DIR}/val.jsonl") else []

SIGNATURES_PATH = f"{DATA_DIR}/signatures/immune_signatures.csv"
gt_df = pd.read_csv(SIGNATURES_PATH, index_col=0) if os.path.exists(SIGNATURES_PATH) else pd.DataFrame()

print(f"Test samples:  {len(test_samples)}")
print(f"Val samples:   {len(val_samples)} (for calibration)")
print(f"GT signatures: {len(gt_df)}")

Test samples:  94
Val samples:   94 (for calibration)
GT signatures: 950


## Inference Engine

In [9]:
MAX_PATCHES = 2  # Must match training config (Phase 5: max_patches=2)
BATCH_SIZE = 2  # For L4: use 2; for A100: use 4

_DEVICE = next((p.device for p in model.parameters() if p.device.type != 'meta'), torch.device('cuda'))
_PAD_TOKEN_ID = processor.tokenizer.eos_token_id


def load_patches_parallel(paths: list, max_patches: int = MAX_PATCHES) -> list:
    paths = paths[:max_patches]
    def _load(p):
        try:
            return Image.open(p).convert("RGB")
        except Exception:
            return None
    with ThreadPoolExecutor(max_workers=min(4, max(1, len(paths)))) as pool:
        return [img for img in pool.map(_load, paths) if img is not None]


def parse_json_output(raw: str) -> dict | None:
    clean = raw
    if '```json' in clean:
        clean = clean.split('```json')[1].split('```')[0]
    elif '```' in clean:
        clean = clean.split('```')[1].split('```')[0]
    s, e = clean.find('{'), clean.rfind('}') + 1
    if s != -1 and e > s:
        try:
            return normalize_prediction_keys(json.loads(clean[s:e]))
        except json.JSONDecodeError:
            pass
    return None


def preload_all_images(samples: list) -> dict:
    """Preload ALL images to RAM — eliminates GDrive I/O during inference."""
    print("  Preloading images to RAM...")
    cache = {}
    all_paths = []
    for s in samples:
        for p in s.get('patch_paths', [])[:MAX_PATCHES]:
            if p not in cache:
                all_paths.append(p)
    def _load(p):
        try:
            return p, Image.open(p).convert("RGB")
        except Exception:
            return p, None
    with ThreadPoolExecutor(max_workers=8) as pool:
        for path, img in tqdm(pool.map(_load, all_paths), total=len(all_paths), desc="Loading patches"):
            if img is not None:
                cache[path] = img
    print(f"  {len(cache)} patches cached in RAM")
    return cache


def prepare_input(sample: dict, image_cache: dict):
    """Prepare single input using cached images."""
    images = [image_cache[p] for p in sample.get('patch_paths', [])[:MAX_PATCHES] if p in image_cache]
    if not images:
        return None, None
    content = [{'type': 'image', 'image': img} for img in images]
    content.append({'type': 'text', 'text': sample['prompt']})
    messages = [{'role': 'user', 'content': content}]
    # do_pan_and_scan=False must match training (Phase 5) to avoid token mismatch
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors='pt',
        do_pan_and_scan=False,
    )
    return inputs, len(images)


def run_batch(batch_samples: list, model, image_cache: dict, max_new_tokens: int = 600) -> list:
    """Run batched inference with left-padding and OOM fallback."""
    prepared = []
    results_skip = []

    for s in batch_samples:
        try:
            inp, n_img = prepare_input(s, image_cache)
            if inp is None:
                results_skip.append({
                    'sample_id': s['sample_id'], 'error': 'no_images',
                    'raw_output': '', 'parsed_json': None, 'json_valid': False, 'n_images': 0
                })
            else:
                prepared.append((inp, s, n_img, inp['input_ids'].shape[-1]))
        except Exception as e:
            results_skip.append({
                'sample_id': s['sample_id'], 'error': str(e),
                'raw_output': '', 'parsed_json': None, 'json_valid': False, 'n_images': 0
            })

    if not prepared:
        return results_skip

    # Single sample — skip padding overhead
    if len(prepared) == 1:
        inp, s, n_img, input_len = prepared[0]
        for k, v in list(inp.items()):
            if torch.is_tensor(v):
                inp[k] = v.to(_DEVICE, dtype=torch.bfloat16) if v.is_floating_point() else v.to(_DEVICE)
        with torch.inference_mode():
            out = model.generate(**inp, max_new_tokens=max_new_tokens,
                                do_sample=False, use_cache=True, pad_token_id=_PAD_TOKEN_ID)
        raw = processor.decode(out[0][input_len:], skip_special_tokens=True).strip()
        parsed = parse_json_output(raw)
        # V3.1: Add cd274_note back (removed from training to save tokens, added in post-processing)
        if parsed and 'cd274_expression' in parsed:
            parsed['cd274_note'] = 'RNA proxy (TCGA RNA-seq); NOT PD-L1 IHC TPS. Confirm with IHC.'
        return results_skip + [{
            'sample_id': s['sample_id'], 'patient_id': s.get('patient_id', ''),
            'raw_output': raw, 'parsed_json': parsed, 'json_valid': parsed is not None,
            'n_images': n_img, 'error': None
        }]

    # Multi-sample: left-pad and batch
    max_len = max(il for _, _, _, il in prepared)
    padded_ids, padded_mask, padded_tti, pv_list = [], [], [], []

    for inp, s, n_img, il in prepared:
        ids = inp['input_ids'].squeeze(0)
        mask = inp['attention_mask'].squeeze(0)
        pad_len = max_len - il

        if pad_len > 0:
            ids = torch.cat([torch.full((pad_len,), _PAD_TOKEN_ID, dtype=ids.dtype), ids])
            mask = torch.cat([torch.zeros(pad_len, dtype=mask.dtype), mask])
        padded_ids.append(ids)
        padded_mask.append(mask)

        if 'token_type_ids' in inp:
            tti = inp['token_type_ids'].squeeze(0)
            if pad_len > 0:
                tti = torch.cat([torch.zeros(pad_len, dtype=tti.dtype), tti])
            padded_tti.append(tti)

        if 'pixel_values' in inp:
            pv_list.append(inp['pixel_values'])

    batched = {
        'input_ids': torch.stack(padded_ids).to(_DEVICE),
        'attention_mask': torch.stack(padded_mask).to(_DEVICE),
    }
    if padded_tti:
        batched['token_type_ids'] = torch.stack(padded_tti).to(_DEVICE)
    if pv_list:
        batched['pixel_values'] = torch.cat(pv_list, dim=0).to(_DEVICE, dtype=torch.bfloat16)

    try:
        with torch.inference_mode():
            output_ids = model.generate(**batched, max_new_tokens=max_new_tokens,
                                       do_sample=False, use_cache=True, pad_token_id=_PAD_TOKEN_ID)
    except (RuntimeError, torch.cuda.OutOfMemoryError):
        torch.cuda.empty_cache()
        print("  Batch OOM — falling back to sequential")
        results = results_skip[:]
        for inp, s, n_img, il in prepared:
            for k, v in list(inp.items()):
                if torch.is_tensor(v):
                    inp[k] = v.to(_DEVICE, dtype=torch.bfloat16) if v.is_floating_point() else v.to(_DEVICE)
            with torch.inference_mode():
                out = model.generate(**inp, max_new_tokens=max_new_tokens,
                                    do_sample=False, use_cache=True, pad_token_id=_PAD_TOKEN_ID)
            raw = processor.decode(out[0][il:], skip_special_tokens=True).strip()
            parsed = parse_json_output(raw)
            results.append({
                'sample_id': s['sample_id'], 'patient_id': s.get('patient_id', ''),
                'raw_output': raw, 'parsed_json': parsed, 'json_valid': parsed is not None,
                'n_images': n_img, 'error': None
            })
        return results

    # Decode batch
    results = results_skip[:]
    for idx, (inp, s, n_img, il) in enumerate(prepared):
        gen_ids = output_ids[idx][max_len:]
        raw = processor.decode(gen_ids, skip_special_tokens=True).strip()
        parsed = parse_json_output(raw)
        results.append({
            'sample_id': s['sample_id'], 'patient_id': s.get('patient_id', ''),
            'raw_output': raw, 'parsed_json': parsed, 'json_valid': parsed is not None,
            'n_images': n_img, 'error': None
        })
    return results


def run_all_inference(samples: list, model, image_cache: dict, label: str = '') -> list:
    preds = []
    times = []
    n = len(samples)
    for i in tqdm(range(0, n, BATCH_SIZE), desc=f'Inference ({label})'):
        batch = samples[i:i+BATCH_SIZE]
        t0 = time.time()
        batch_results = run_batch(batch, model, image_cache)
        elapsed = time.time() - t0
        per_sample = elapsed / len(batch)
        for r in batch_results:
            r['inference_time_s'] = round(per_sample, 2)
            preds.append(r)
            times.append(per_sample)

    n_valid = sum(1 for p in preds if p['json_valid'])
    print(f'  {label}: {n_valid}/{len(preds)} valid JSON ({n_valid/max(1,len(preds))*100:.0f}%), '
          f'avg {np.mean(times):.1f}s/sample, total {sum(times)/60:.1f}min')
    return preds


print("Inference engine ready")
print(f"  MAX_PATCHES: {MAX_PATCHES}")
print(f"  BATCH_SIZE:  {BATCH_SIZE}")
print(f"  Device:      {_DEVICE}")

Inference engine ready
  MAX_PATCHES: 2
  BATCH_SIZE:  2
  Device:      cuda:0


## Run Inference on Test + Val Sets

In [10]:
# Preload images once
all_samples = test_samples + val_samples
image_cache = preload_all_images(all_samples)

print('\nRunning inference on test set...')
test_preds = run_all_inference(test_samples, model, image_cache, 'test')

print('\nRunning inference on val set...')
val_preds = run_all_inference(val_samples, model, image_cache, 'val')

# Free image cache
del image_cache
gc.collect()

# Save predictions
def save_predictions(preds, path):
    with open(path, 'w') as f:
        for p in preds:
            f.write(json.dumps(p, default=str) + '\n')

save_predictions(test_preds, f"{RESULTS_DIR}/finetuned_test_predictions.jsonl")
save_predictions(val_preds, f"{RESULTS_DIR}/finetuned_val_predictions.jsonl")
print(f"\nPredictions saved to {RESULTS_DIR}")

  Preloading images to RAM...


Loading patches:   0%|          | 0/375 [00:00<?, ?it/s]

  375 patches cached in RAM

Running inference on test set...


Inference (test):   0%|          | 0/47 [00:00<?, ?it/s]

  test: 94/94 valid JSON (100%), avg 12.3s/sample, total 19.2min

Running inference on val set...


Inference (val):   0%|          | 0/47 [00:00<?, ?it/s]

  val: 94/94 valid JSON (100%), avg 12.3s/sample, total 19.3min

Predictions saved to /content/drive/MyDrive/ImmunoPath/results/phase6_v3.1


## ImmunoPathEvaluator — Full Metrics Suite

In [11]:
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score, f1_score, confusion_matrix,
    mean_absolute_error, brier_score_loss,
)
from scipy.stats import spearmanr, pearsonr


class ImmunoPathEvaluator:
    """Compute all evaluation metrics from the spec.

    NOTE on binary metrics (CD274, MSI):
    With hard {0,1} predictions (no probabilities), we compute balanced accuracy
    = (TPR + TNR) / 2, NOT true AUC which requires continuous probability scores.
    Previously mislabeled as 'AUC' in v2/v3.0 (fixed in v3.1).
    """

    TME_LABELS = {"IE": 0, "IE/F": 1, "F": 2, "D": 3}

    def __init__(self, predictions: list, eval_samples: list, ground_truth_df: pd.DataFrame):
        self.predictions = predictions
        self.eval_samples = eval_samples
        self.gt_df = ground_truth_df
        self._sample_map = {s["sample_id"]: s for s in eval_samples}

    def _get_ground_truth_response(self, sample_id: str) -> dict:
        s = self._sample_map.get(sample_id)
        if s and "response" in s:
            try:
                return json.loads(s["response"])
            except Exception:
                return {}
        return {}

    def evaluate_cd274(self) -> dict:
        """CD274 binary classification.

        NOTE: With hard {0,1} predictions (no probabilities), we compute
        balanced accuracy = (TPR + TNR) / 2, NOT true AUC which requires
        continuous probability scores. Previously mislabeled as 'AUC'.
        """
        y_true, y_pred = [], []
        for p in self.predictions:
            if not p.get("json_valid") or not p.get("parsed_json"):
                continue
            gt = self._get_ground_truth_response(p["sample_id"])
            true_val = str(gt.get("cd274_expression", "")).lower()
            pred_val = str(p["parsed_json"].get("cd274_expression", "")).lower()
            if true_val in ("high", "low") and pred_val in ("high", "low"):
                y_true.append(1 if true_val == "high" else 0)
                y_pred.append(1 if pred_val == "high" else 0)

        result = {"n_samples": len(y_true)}
        if len(y_true) >= 2 and len(set(y_true)) > 1:
            # balanced_accuracy_score = (TPR + TNR) / 2 — correct for hard predictions
            result["balanced_accuracy"] = round(balanced_accuracy_score(y_true, y_pred), 4)
            result["accuracy"] = round(accuracy_score(y_true, y_pred), 4)
            result["f1"] = round(f1_score(y_true, y_pred, zero_division=0), 4)
            result["brier"] = round(brier_score_loss(y_true, y_pred), 4)
        elif y_true:
            result["accuracy"] = round(accuracy_score(y_true, y_pred), 4)
        return result

    def evaluate_msi(self) -> dict:
        """MSI binary classification.

        WARNING: MSI-H prevalence in TCGA NSCLC is ~0.58% (~4 unique patients).
        With so few positives, this metric is statistically unreliable.
        Balanced accuracy with hard predictions (not true AUC).
        """
        y_true, y_pred = [], []
        for p in self.predictions:
            if not p.get("json_valid") or not p.get("parsed_json"):
                continue
            gt = self._get_ground_truth_response(p["sample_id"])
            true_msi = str(gt.get("msi_status", "")).upper()
            pred_msi = str(p["parsed_json"].get("msi_status", "")).upper()
            if true_msi in ("MSI-H", "MSS") and pred_msi in ("MSI-H", "MSS"):
                y_true.append(1 if true_msi == "MSI-H" else 0)
                y_pred.append(1 if pred_msi == "MSI-H" else 0)

        result = {"n_samples": len(y_true)}
        n_positive = sum(y_true)
        result["n_msi_h"] = n_positive
        if n_positive < 3:
            result["warning"] = f"Only {n_positive} MSI-H samples — metric statistically unreliable"
        if len(y_true) >= 2 and len(set(y_true)) > 1:
            result["balanced_accuracy"] = round(balanced_accuracy_score(y_true, y_pred), 4)
            result["accuracy"] = round(accuracy_score(y_true, y_pred), 4)
            result["f1"] = round(f1_score(y_true, y_pred, zero_division=0), 4)
        elif y_true:
            result["accuracy"] = round(accuracy_score(y_true, y_pred), 4)
        return result

    def evaluate_tme(self) -> dict:
        y_true, y_pred = [], []
        for p in self.predictions:
            if not p.get("json_valid") or not p.get("parsed_json"):
                continue
            gt = self._get_ground_truth_response(p["sample_id"])
            true_tme = str(gt.get("tme_subtype", ""))
            pred_tme = str(p["parsed_json"].get("tme_subtype", "")).replace("IE_F", "IE/F").replace("IE-F", "IE/F")
            if true_tme in self.TME_LABELS and pred_tme in self.TME_LABELS:
                y_true.append(self.TME_LABELS[true_tme])
                y_pred.append(self.TME_LABELS[pred_tme])

        result = {"n_samples": len(y_true)}
        if y_true:
            result["accuracy"] = round(accuracy_score(y_true, y_pred), 4)
            result["macro_f1"] = round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4)
            result["weighted_f1"] = round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4)
            labels = list(range(len(self.TME_LABELS)))
            cm = confusion_matrix(y_true, y_pred, labels=labels)
            result["confusion_matrix"] = cm.tolist()
            result["class_names"] = list(self.TME_LABELS.keys())
        return result

    def evaluate_til(self) -> dict:
        y_true, y_pred = [], []
        for p in self.predictions:
            if not p.get("json_valid") or not p.get("parsed_json"):
                continue
            gt = self._get_ground_truth_response(p["sample_id"])
            try:
                true_val = float(gt.get("til_fraction"))
                pred_val = float(p["parsed_json"].get("til_fraction"))
                y_true.append(true_val)
                y_pred.append(pred_val)
            except (ValueError, TypeError):
                pass

        result = {"n_samples": len(y_true)}
        if len(y_true) >= 3:
            rho, pval = spearmanr(y_true, y_pred)
            r, r_pval = pearsonr(y_true, y_pred)
            result["spearman_rho"] = round(rho, 4)
            result["spearman_p"] = round(pval, 6)
            result["pearson_r"] = round(r, 4)
            result["mae"] = round(mean_absolute_error(y_true, y_pred), 4)
        return result

    def evaluate_immune_score(self) -> dict:
        y_true, y_pred = [], []
        for p in self.predictions:
            if not p.get("json_valid") or not p.get("parsed_json"):
                continue
            gt = self._get_ground_truth_response(p["sample_id"])
            try:
                true_val = float(gt.get("immune_score"))
                pred_val = float(p["parsed_json"].get("immune_score"))
                y_true.append(true_val)
                y_pred.append(pred_val)
            except (ValueError, TypeError):
                pass

        result = {"n_samples": len(y_true)}
        if len(y_true) >= 3:
            rho, _ = spearmanr(y_true, y_pred)
            result["spearman_rho"] = round(rho, 4)
            result["mae"] = round(mean_absolute_error(y_true, y_pred), 4)
        return result

    def evaluate_json(self) -> dict:
        n_total = len(self.predictions)
        n_valid = sum(1 for p in self.predictions if p.get("json_valid"))
        n_errors = sum(1 for p in self.predictions if p.get("error"))
        # V3.1: 8 core fields (removed prediction_entropy, low_confidence_flag, cd274_note)
        REQUIRED_KEYS = [
            "cd274_expression", "msi_status", "tme_subtype", "til_fraction",
            "til_density", "immune_phenotype", "cd8_infiltration", "immune_score",
        ]
        n_schema_ok = 0
        for p in self.predictions:
            if p.get("json_valid") and p.get("parsed_json"):
                if all(k in p["parsed_json"] for k in REQUIRED_KEYS):
                    n_schema_ok += 1
        return {
            "n_total": n_total,
            "n_valid_json": n_valid,
            "json_parse_rate": round(n_valid / max(1, n_total), 4),
            "n_schema_compliant": n_schema_ok,
            "schema_compliance_rate": round(n_schema_ok / max(1, n_total), 4),
            "n_errors": n_errors,
        }

    @staticmethod
    def compute_ece(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
        y_true = np.asarray(y_true)
        y_prob = np.asarray(y_prob)
        bin_boundaries = np.linspace(0, 1, n_bins + 1)
        ece = 0.0
        for i in range(n_bins):
            mask = (y_prob >= bin_boundaries[i]) & (y_prob < bin_boundaries[i + 1])
            if mask.sum() > 0:
                bin_acc = y_true[mask].mean()
                bin_conf = y_prob[mask].mean()
                ece += (mask.sum() / len(y_true)) * abs(bin_acc - bin_conf)
        return round(ece, 4)

    def evaluate_all(self) -> dict:
        return {
            "cd274": self.evaluate_cd274(),
            "msi": self.evaluate_msi(),
            "tme": self.evaluate_tme(),
            "til": self.evaluate_til(),
            "immune_score": self.evaluate_immune_score(),
            "json": self.evaluate_json(),
        }


# --- Run Evaluation ---
evaluator = ImmunoPathEvaluator(test_preds, test_samples, gt_df)
ft_metrics = evaluator.evaluate_all()

print("\n" + "=" * 60)
print("FINE-TUNED MODEL METRICS (v3.1 — Test Set)")
print("=" * 60)
for task, m in ft_metrics.items():
    print(f"\n  {task}:")
    for k, v in m.items():
        if k != "confusion_matrix":
            print(f"    {k}: {v}")


FINE-TUNED MODEL METRICS (v3.1 — Test Set)

  cd274:
    n_samples: 94
    balanced_accuracy: 0.5106
    accuracy: 0.5106
    f1: 0.0417
    brier: 0.4894

  msi:
    n_samples: 94
    n_msi_h: 0
    accuracy: 1.0

  tme:
    n_samples: 91
    accuracy: 0.2747
    macro_f1: 0.1078
    weighted_f1: 0.1232
    class_names: ['IE', 'IE/F', 'F', 'D']

  til:
    n_samples: 94
    spearman_rho: 0.1194
    spearman_p: 0.251671
    pearson_r: -0.1223
    mae: 0.1615

  immune_score:
    n_samples: 94
    spearman_rho: -0.0721
    mae: 0.2071

  json:
    n_total: 94
    n_valid_json: 94
    json_parse_rate: 1.0
    n_schema_compliant: 94
    schema_compliance_rate: 1.0
    n_errors: 0


## Temperature Scaling Calibration

In [12]:
from scipy.optimize import minimize_scalar

def find_temperature(val_preds: list, val_samples: list) -> float:
    # NOTE (v3.1): Temperature scaling uses heuristic confidence (0.8/0.2),
    # NOT actual model probabilities. ECE metrics are approximate only.
    # For true calibration, extract logits from model.generate() in future work.
    """Learn temperature T on validation set for CD274 binary predictions.

    NOTE (v3.1): Uses heuristic confidence (0.8/0.2), NOT actual model logits.
    ECE/Brier metrics are approximate placeholders. For true calibration,
    extract token logits from model.generate() in future work.
    """
    y_true, y_conf = [], []
    sample_map = {s["sample_id"]: s for s in val_samples}

    for p in val_preds:
        if not p.get("json_valid") or not p.get("parsed_json"):
            continue
        s = sample_map.get(p["sample_id"])
        if not s:
            continue
        try:
            gt = json.loads(s["response"])
        except Exception:
            continue
        true_cd274 = str(gt.get("cd274_expression", "")).lower()
        pred_cd274 = str(p["parsed_json"].get("cd274_expression", "")).lower()
        if true_cd274 in ("high", "low") and pred_cd274 in ("high", "low"):
            y_true.append(1.0 if true_cd274 == "high" else 0.0)
            y_conf.append(0.8 if pred_cd274 == true_cd274 else 0.2)

    if len(y_true) < 5:
        print("Not enough samples for temperature scaling — using T=1.0")
        return 1.0

    y_true = np.array(y_true)
    y_conf = np.array(y_conf)
    logits = np.log(y_conf / (1 - y_conf + 1e-8))

    def nll(T):
        scaled = 1.0 / (1.0 + np.exp(-logits / T))
        scaled = np.clip(scaled, 1e-8, 1 - 1e-8)
        return -np.mean(y_true * np.log(scaled) + (1 - y_true) * np.log(1 - scaled))

    result = minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded')
    T = result.x
    print(f"Optimal temperature: T = {T:.3f}")
    return T


temperature = find_temperature(val_preds, val_samples)

# Compute ECE before and after calibration on test set
cd274_true_a = None
cd274_conf_a = None
calibrated_conf = None
calibration_metrics = {"temperature": round(temperature, 4)}

cd274_true_list, cd274_conf_list = [], []
sample_map_test = {s["sample_id"]: s for s in test_samples}
for p in test_preds:
    if not p.get("json_valid") or not p.get("parsed_json"):
        continue
    s = sample_map_test.get(p["sample_id"])
    if not s:
        continue
    try:
        gt = json.loads(s["response"])
    except Exception:
        continue
    true_cd274 = str(gt.get("cd274_expression", "")).lower()
    pred_cd274 = str(p["parsed_json"].get("cd274_expression", "")).lower()
    if true_cd274 in ("high", "low") and pred_cd274 in ("high", "low"):
        cd274_true_list.append(1 if true_cd274 == "high" else 0)
        cd274_conf_list.append(0.8 if pred_cd274 == true_cd274 else 0.2)

if len(cd274_true_list) >= 5:
    cd274_true_a = np.array(cd274_true_list)
    cd274_conf_a = np.array(cd274_conf_list)
    logits_test = np.log(cd274_conf_a / (1 - cd274_conf_a + 1e-8))
    calibrated_conf = 1.0 / (1.0 + np.exp(-logits_test / temperature))

    calibration_metrics["ece_before"] = ImmunoPathEvaluator.compute_ece(cd274_true_a, cd274_conf_a)
    calibration_metrics["ece_after"] = ImmunoPathEvaluator.compute_ece(cd274_true_a, calibrated_conf)
    calibration_metrics["brier_before"] = round(brier_score_loss(cd274_true_a, cd274_conf_a), 4)
    calibration_metrics["brier_after"] = round(brier_score_loss(cd274_true_a, calibrated_conf), 4)

    print(f"\n  ECE before calibration: {calibration_metrics['ece_before']}")
    print(f"  ECE after calibration:  {calibration_metrics['ece_after']}")
    print(f"  Brier before: {calibration_metrics['brier_before']}")
    print(f"  Brier after:  {calibration_metrics['brier_after']}")
else:
    print(f"Only {len(cd274_true_list)} CD274 samples — not enough for calibration")

Optimal temperature: T = 10.000

  ECE before calibration: 0.7894
  ECE after calibration:  0.524
  Brier before: 0.6336
  Brier after:  0.2851


## Bootstrap 95% Confidence Intervals

In [13]:
def bootstrap_metric(y_true, y_pred, metric_fn, n_bootstrap=1000, seed=42):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    if n < 3:
        return None, None, None
    point = metric_fn(y_true, y_pred)
    boot_values = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        try:
            val = metric_fn(y_true[idx], y_pred[idx])
            if np.isfinite(val):
                boot_values.append(val)
        except Exception:
            pass
    if len(boot_values) < 100:
        return point, None, None
    lower = np.percentile(boot_values, 2.5)
    upper = np.percentile(boot_values, 97.5)
    return round(point, 4), round(lower, 4), round(upper, 4)


bootstrap_results = {}

# CD274 AUC CI
cd274_true_b, cd274_pred_b = [], []
for p in test_preds:
    if not p.get("json_valid") or not p.get("parsed_json"):
        continue
    gt = evaluator._get_ground_truth_response(p["sample_id"])
    true_v = str(gt.get("cd274_expression", "")).lower()
    pred_v = str(p["parsed_json"].get("cd274_expression", "")).lower()
    if true_v in ("high", "low") and pred_v in ("high", "low"):
        cd274_true_b.append(1 if true_v == "high" else 0)
        cd274_pred_b.append(1 if pred_v == "high" else 0)

if len(cd274_true_b) >= 5 and len(set(cd274_true_b)) > 1:
    pt, lo, hi = bootstrap_metric(cd274_true_b, cd274_pred_b, balanced_accuracy_score)
    bootstrap_results["cd274_balanced_accuracy"] = {"point": pt, "ci_lower": lo, "ci_upper": hi}
    print(f"  CD274 Balanced Acc: {pt} [{lo}, {hi}]")

# MSI AUC CI
msi_true_b, msi_pred_b = [], []
for p in test_preds:
    if not p.get("json_valid") or not p.get("parsed_json"):
        continue
    gt = evaluator._get_ground_truth_response(p["sample_id"])
    true_v = str(gt.get("msi_status", "")).upper()
    pred_v = str(p["parsed_json"].get("msi_status", "")).upper()
    if true_v in ("MSI-H", "MSS") and pred_v in ("MSI-H", "MSS"):
        msi_true_b.append(1 if true_v == "MSI-H" else 0)
        msi_pred_b.append(1 if pred_v == "MSI-H" else 0)

if len(msi_true_b) >= 5 and len(set(msi_true_b)) > 1:
    pt, lo, hi = bootstrap_metric(msi_true_b, msi_pred_b, balanced_accuracy_score)
    bootstrap_results["msi_balanced_accuracy"] = {"point": pt, "ci_lower": lo, "ci_upper": hi}
    print(f"  MSI Balanced Acc: {pt} [{lo}, {hi}]")

# TME Accuracy CI
tme_true_b, tme_pred_b = [], []
for p in test_preds:
    if not p.get("json_valid") or not p.get("parsed_json"):
        continue
    gt = evaluator._get_ground_truth_response(p["sample_id"])
    true_tme = str(gt.get("tme_subtype", ""))
    pred_tme = str(p["parsed_json"].get("tme_subtype", "")).replace("IE_F", "IE/F").replace("IE-F", "IE/F")
    if true_tme in evaluator.TME_LABELS and pred_tme in evaluator.TME_LABELS:
        tme_true_b.append(evaluator.TME_LABELS[true_tme])
        tme_pred_b.append(evaluator.TME_LABELS[pred_tme])

if len(tme_true_b) >= 5:
    pt, lo, hi = bootstrap_metric(tme_true_b, tme_pred_b, accuracy_score)
    bootstrap_results["tme_accuracy"] = {"point": pt, "ci_lower": lo, "ci_upper": hi}
    print(f"  TME Accuracy: {pt} [{lo}, {hi}]")

# TIL Spearman CI
til_true_b, til_pred_b = [], []
for p in test_preds:
    if not p.get("json_valid") or not p.get("parsed_json"):
        continue
    gt = evaluator._get_ground_truth_response(p["sample_id"])
    try:
        tv = float(gt.get("til_fraction"))
        pv = float(p["parsed_json"].get("til_fraction"))
        til_true_b.append(tv)
        til_pred_b.append(pv)
    except (ValueError, TypeError):
        pass

if len(til_true_b) >= 5:
    def spearman_fn(y_true, y_pred):
        return spearmanr(y_true, y_pred)[0]
    pt, lo, hi = bootstrap_metric(til_true_b, til_pred_b, spearman_fn)
    bootstrap_results["til_spearman"] = {"point": pt, "ci_lower": lo, "ci_upper": hi}
    print(f"  TIL Spearman: {pt} [{lo}, {hi}]")

print(f"\nBootstrap CIs computed for {len(bootstrap_results)} metrics")

  CD274 Balanced Acc: 0.5106 [0.5, 0.5349]
  TME Accuracy: 0.2747 [0.1868, 0.3736]
  TIL Spearman: 0.1194 [-0.0766, 0.3131]

Bootstrap CIs computed for 3 metrics


## Load Phase 4 Zero-Shot + Compare

In [14]:
# Load zero-shot predictions from Phase 4
zs_metrics = {}
zs_path = f"{PHASE4_DIR}/zero_shot_predictions.jsonl"

if os.path.exists(zs_path):
    zs_preds = []
    with open(zs_path) as f:
        for line in f:
            if not line.strip():
                continue
            p = json.loads(line.strip())
            if p.get("parsed_json"):
                p["parsed_json"] = normalize_prediction_keys(p["parsed_json"])
                p["json_valid"] = True
            zs_preds.append(p)

    # Match zero-shot preds to test samples
    test_ids = {s["sample_id"] for s in test_samples}
    zs_test = [p for p in zs_preds if p["sample_id"] in test_ids]

    if len(zs_test) < 5:
        all_samples = test_samples + val_samples
        all_map = {s["sample_id"]: s for s in all_samples}
        zs_test = [p for p in zs_preds if p["sample_id"] in all_map]
        eval_samples_for_zs = [all_map[p["sample_id"]] for p in zs_test if p["sample_id"] in all_map]
        print(f"Phase 4 IDs differ — matched {len(zs_test)} predictions")
        if eval_samples_for_zs:
            zs_metrics = ImmunoPathEvaluator(zs_test, eval_samples_for_zs, gt_df).evaluate_all()
    else:
        zs_metrics = ImmunoPathEvaluator(zs_test, test_samples, gt_df).evaluate_all()

    print(f"Zero-shot: {len(zs_test)} matched predictions")
    for t, m in zs_metrics.items():
        print(f"  {t}: {m}")
else:
    print(f"Zero-shot predictions not found at {zs_path}")
    print("Comparison will show N/A for zero-shot metrics")

# === Comparison Table ===
tasks = [
    ("CD274 Bal. Acc", "cd274", "balanced_accuracy", ">0.70"),
    ("CD274 Accuracy", "cd274", "accuracy", None),
    ("MSI Bal. Acc", "msi", "balanced_accuracy", ">0.75"),
    ("MSI Accuracy", "msi", "accuracy", None),
    ("TME Accuracy", "tme", "accuracy", ">0.65"),
    ("TME Macro-F1", "tme", "macro_f1", None),
    ("TIL Spearman \u03c1", "til", "spearman_rho", ">0.60"),
    ("TIL MAE", "til", "mae", "<0.20"),
    ("Immune Score MAE", "immune_score", "mae", "<0.15"),
    ("JSON Parse Rate", "json", "json_parse_rate", None),
]

print("\n" + "=" * 80)
print("ZERO-SHOT vs FINE-TUNED (v3.1) COMPARISON")
print("=" * 80)
print(f"{'Task':<25}{'Zero-Shot':>12}{'Fine-Tuned':>12}{'Delta':>10}{'Target':>10}")
print("-" * 80)
for name, tk, mk, tgt in tasks:
    zv = zs_metrics.get(tk, {}).get(mk)
    fv = ft_metrics.get(tk, {}).get(mk)
    zs = f"{zv:.4f}" if isinstance(zv, (int, float)) else "N/A"
    fs = f"{fv:.4f}" if isinstance(fv, (int, float)) else "N/A"
    d = f"{fv - zv:+.4f}" if isinstance(zv, (int, float)) and isinstance(fv, (int, float)) else ""
    st = ""
    if tgt and isinstance(fv, (int, float)):
        if tgt[0] == ">":
            st = "\u2705" if fv > float(tgt[1:]) else "\u274c"
        elif tgt[0] == "<":
            st = "\u2705" if fv < float(tgt[1:]) else "\u274c"
    print(f"  {name:<23}{zs:>12}{fs:>12}{d:>10}{(tgt or ''):>10} {st}")

Zero-shot: 14 matched predictions
  cd274: {'n_samples': 11, 'balanced_accuracy': np.float64(0.75), 'accuracy': 0.7273, 'f1': 0.7692, 'brier': np.float64(0.2727)}
  msi: {'n_samples': 11, 'n_msi_h': 0, 'warning': 'Only 0 MSI-H samples — metric statistically unreliable', 'accuracy': 0.2727}
  tme: {'n_samples': 11, 'accuracy': 0.3636, 'macro_f1': 0.1333, 'weighted_f1': 0.1939, 'confusion_matrix': [[4, 0, 0, 0], [2, 0, 0, 0], [3, 0, 0, 0], [2, 0, 0, 0]], 'class_names': ['IE', 'IE/F', 'F', 'D']}
  til: {'n_samples': 11, 'spearman_rho': np.float64(-0.1291), 'spearman_p': np.float64(0.705201), 'pearson_r': np.float64(-0.1169), 'mae': 0.3291}
  immune_score: {'n_samples': 11, 'spearman_rho': np.float64(-0.5117), 'mae': 0.2275}
  json: {'n_total': 14, 'n_valid_json': 12, 'json_parse_rate': 0.8571, 'n_schema_compliant': 2, 'schema_compliance_rate': 0.1429, 'n_errors': 0}

ZERO-SHOT vs FINE-TUNED (v3.1) COMPARISON
Task                        Zero-Shot  Fine-Tuned     Delta    Target
-----------

## Generate Plots

In [15]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)

# Plot 1: TME Confusion Matrix
tme_result = ft_metrics.get("tme", {})
if "confusion_matrix" in tme_result:
    fig, ax = plt.subplots(figsize=(7, 6))
    cm = np.array(tme_result["confusion_matrix"])
    class_names = tme_result.get("class_names", list(ImmunoPathEvaluator.TME_LABELS.keys()))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted TME Subtype")
    ax.set_ylabel("True TME Subtype")
    ax.set_title("TME Subtype Confusion Matrix (Fine-Tuned v3.1)")
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/confusion_matrix_tme.png", dpi=150)
    plt.close(fig)
    print("TME confusion matrix saved")

# Plot 2: Calibration Curve
if cd274_true_a is not None and cd274_conf_a is not None and calibrated_conf is not None:
    fig, ax = plt.subplots(figsize=(7, 6))
    n_bins = 10
    bins = np.linspace(0, 1, n_bins + 1)

    bin_accs_before, bin_confs_before = [], []
    for i in range(n_bins):
        mask = (cd274_conf_a >= bins[i]) & (cd274_conf_a < bins[i + 1])
        if mask.sum() > 0:
            bin_accs_before.append(cd274_true_a[mask].mean())
            bin_confs_before.append(cd274_conf_a[mask].mean())

    bin_accs_after, bin_confs_after = [], []
    for i in range(n_bins):
        mask = (calibrated_conf >= bins[i]) & (calibrated_conf < bins[i + 1])
        if mask.sum() > 0:
            bin_accs_after.append(cd274_true_a[mask].mean())
            bin_confs_after.append(calibrated_conf[mask].mean())

    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    if bin_confs_before:
        ax.plot(bin_confs_before, bin_accs_before, 'o-',
                label=f'Before (ECE={calibration_metrics.get("ece_before", "?")})')
    if bin_confs_after:
        ax.plot(bin_confs_after, bin_accs_after, 's-',
                label=f'After T={temperature:.2f} (ECE={calibration_metrics.get("ece_after", "?")})')
    ax.set_xlabel("Mean Predicted Confidence")
    ax.set_ylabel("Fraction of Positives")
    ax.set_title("CD274 Calibration Curve (v3.1)")
    ax.legend()
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/calibration_curves.png", dpi=150)
    plt.close(fig)
    print("Calibration curves saved")

# Plot 3: Zero-Shot vs Fine-Tuned Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
metric_names = []
zs_values = []
ft_values = []
for task_name, task_key, metric_key, _ in tasks[:7]:
    zs_v = zs_metrics.get(task_key, {}).get(metric_key)
    ft_v = ft_metrics.get(task_key, {}).get(metric_key)
    if isinstance(zs_v, (int, float)) and isinstance(ft_v, (int, float)):
        metric_names.append(task_name)
        zs_values.append(zs_v)
        ft_values.append(ft_v)

if metric_names:
    x = np.arange(len(metric_names))
    width = 0.35
    ax.bar(x - width / 2, zs_values, width, label='Zero-Shot', color='#93c5fd')
    ax.bar(x + width / 2, ft_values, width, label='Fine-Tuned v3.1', color='#22c55e')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names, rotation=30, ha='right')
    ax.set_ylabel("Score")
    ax.set_title("Zero-Shot vs Fine-Tuned v3.1 Performance")
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/zero_shot_vs_finetuned.png", dpi=150)
    plt.close(fig)
    print("Comparison bar chart saved")

TME confusion matrix saved
Calibration curves saved
Comparison bar chart saved


## Save Full Evaluation Report

In [16]:
from datetime import datetime

full_report = {
    "phase": "6_v3.1",
    "timestamp": datetime.now().isoformat(),
    "model_id": MODEL_ID,
    "adapter_dir": ADAPTER_DIR,
    "training_version": "v3.1",
    "v3_1_fixes_applied": [
        "Response-only loss masking (prompt + turn markers masked)",
        "REMOVED modules_to_save (avoids PEFT tied weight corruption)",
        "3 epochs + EarlyStoppingCallback(patience=2)",
        "LR 1e-4 (was 2e-4 — preserves pretrained features)",
        "V3.1: til_fraction fixed (was =immune_score, now =normalized til_signature)",
        "V3.1: Removed noisy fields (prediction_entropy, low_confidence_flag, cd274_note)",
        "V3.1: MSI-H oversampling DISABLED (4 patients = memorization)",
        "V3.1: TME 'unknown' excluded from oversampling (not in eval metrics)",
        "V3.1: Stratified patient-level split (was random shuffle)",
    ],
    "metric_notes": {
        "cd274_balanced_accuracy": "Balanced accuracy (TPR+TNR)/2 from hard predictions, NOT true AUC",
        "msi_balanced_accuracy": "Balanced accuracy from hard predictions; only ~4 MSI-H patients — unreliable",
    },
    "n_test_samples": len(test_samples),
    "n_val_samples": len(val_samples),
    "finetuned_metrics": ft_metrics,
    "zero_shot_metrics": zs_metrics,
    "calibration": calibration_metrics,
    "bootstrap_cis": bootstrap_results,
}

with open(f"{RESULTS_DIR}/evaluation_results.json", 'w') as f:
    json.dump(full_report, f, indent=2, default=str)
print(f"Evaluation results saved to {RESULTS_DIR}/evaluation_results.json")

# Generate comparison markdown
md_lines = [
    "# Zero-Shot vs Fine-Tuned v3.1 Comparison\n",
    f"**Date:** {datetime.now().strftime('%Y-%m-%d')}\n",
    f"**Model:** {MODEL_ID} + LoRA (r=16, α=16)\n",
    f"**Training:** SFTTrainer, 3 epochs + early stopping (patience=2), response-only loss, LR=1e-4\n",
    f"**Test samples:** {len(test_samples)}\n",
    "**NOTE:** CD274/MSI metrics are balanced accuracy (hard predictions), not true AUC.\n",
    "",
    "| Task | Zero-Shot | Fine-Tuned v3.1 | Delta | Target |",
    "|------|-----------|-----------------|-------|--------|",
]
for task_name, task_key, metric_key, target in tasks:
    zs_val = zs_metrics.get(task_key, {}).get(metric_key)
    ft_val = ft_metrics.get(task_key, {}).get(metric_key)
    zs_str = f"{zs_val:.4f}" if isinstance(zs_val, (int, float)) else "N/A"
    ft_str = f"{ft_val:.4f}" if isinstance(ft_val, (int, float)) else "N/A"
    delta = ""
    if isinstance(zs_val, (int, float)) and isinstance(ft_val, (int, float)):
        d = ft_val - zs_val
        delta = f"{d:+.4f}"
    t = target if target else ""
    md_lines.append(f"| {task_name} | {zs_str} | {ft_str} | {delta} | {t} |")

md_lines.extend([
    "",
    "## Calibration",
    f"- Temperature: {calibration_metrics.get('temperature', 'N/A')}",
    f"- ECE before: {calibration_metrics.get('ece_before', 'N/A')}",
    f"- ECE after: {calibration_metrics.get('ece_after', 'N/A')}",
    "",
    "## Bootstrap 95% CIs",
])
for metric_name, ci in bootstrap_results.items():
    md_lines.append(f"- {metric_name}: {ci['point']} [{ci.get('ci_lower', '?')}, {ci.get('ci_upper', '?')}]")

md_lines.extend([
    "",
    "## Key Fixes (v2 → v3.1)",
    "1. **Response-only loss masking** — 100% of gradient on prediction values (was 9%)",
    "2. **REMOVED modules_to_save** — fixes PEFT #2777 tied weight corruption",
    "3. **3 epochs + early stopping** — with patience=2",
    "4. **LR 1e-4** — preserves pretrained histopathology features (was 2e-4)",
    "5. **MSI-H oversampling DISABLED** — 4 MSI-H patients = memorization, not learning",
    "",
    "## Metric Notes",
    "- CD274/MSI 'Bal. Acc' = balanced accuracy from hard {0,1} predictions, NOT true AUC",
    "- MSI-H prevalence ~0.58% in TCGA NSCLC (~4 patients) — metric statistically unreliable",
    "- Temperature scaling uses heuristic 0.8/0.2 confidence, not actual model logits",
])

with open(f"{RESULTS_DIR}/zero_shot_vs_finetuned_v3_1.md", 'w') as f:
    f.write("\n".join(md_lines))
print(f"Comparison report saved")

Evaluation results saved to /content/drive/MyDrive/ImmunoPath/results/phase6_v3.1/evaluation_results.json
Comparison report saved


## Phase 6 v3.1 Summary

In [17]:
print("\n" + "=" * 60)
print("PHASE 6 v3.1 — EVALUATION COMPLETE")
print("=" * 60)

print(f"\n  Model: {MODEL_ID} + LoRA v3.1 adapters")
print(f"  Test samples: {len(test_samples)}")

print(f"\n  --- Key Metrics (Fine-Tuned v3.1) ---")
cd274_bal_acc = ft_metrics.get("cd274", {}).get("balanced_accuracy", "N/A")
msi_bal_acc = ft_metrics.get("msi", {}).get("balanced_accuracy", "N/A")
tme_acc = ft_metrics.get("tme", {}).get("accuracy", "N/A")
til_rho = ft_metrics.get("til", {}).get("spearman_rho", "N/A")
iscore_mae = ft_metrics.get("immune_score", {}).get("mae", "N/A")
json_rate = ft_metrics.get("json", {}).get("json_parse_rate", "N/A")
msi_warning = ft_metrics.get("msi", {}).get("warning", "")
msi_n_h = ft_metrics.get("msi", {}).get("n_msi_h", "?")

print(f"  CD274 Bal. Acc:    {cd274_bal_acc}  (target: >0.70)")
print(f"  MSI Bal. Acc:      {msi_bal_acc}  (target: >0.75)")
if msi_warning:
    print(f"    ⚠ {msi_warning}")
print(f"  TME Accuracy:      {tme_acc}  (target: >0.65)")
print(f"  TIL Spearman ρ:    {til_rho}  (target: >0.60)")
print(f"  Immune Score MAE:  {iscore_mae}  (target: <0.15)")
print(f"  JSON Parse Rate:   {json_rate}")

print(f"\n  --- Calibration ---")
print(f"  Temperature: {calibration_metrics.get('temperature', 'N/A')}")
print(f"  ECE before:  {calibration_metrics.get('ece_before', 'N/A')}  (target: <0.10)")
print(f"  ECE after:   {calibration_metrics.get('ece_after', 'N/A')}")

# v2 comparison
print(f"\n  --- v2 Metrics (for reference) ---")
print(f"  v2 CD274 Bal. Acc: 0.5638  → v3.1: {cd274_bal_acc}")
print(f"  v2 MSI Bal. Acc:   0.5000  → v3.1: {msi_bal_acc}  (⚠ {msi_n_h} MSI-H patients in test)")
print(f"  v2 TME Accuracy:   0.2473  → v3.1: {tme_acc}")
print(f"  v2 TIL Spearman:   -0.1740 → v3.1: {til_rho}")
print(f"  v2 Immune MAE:     0.3032  → v3.1: {iscore_mae}")

print(f"\n  --- V3.1 Fixes Applied ---")
print(f"  1. Response-only loss masking ✓")
print(f"  2. No modules_to_save (weight tying preserved) ✓")
print(f"  3. 3 epochs + early stopping (patience=2) ✓")
print(f"  4. LR 1e-4 (was 2e-4) ✓")
print(f"  5. MSI-H oversampling DISABLED (4 patients = memorization) ✓")
print(f"  NOTE: CD274/MSI use balanced accuracy (hard predictions), not true AUC")

print(f"\n{'=' * 60}")
print("Saved to:")
print(f"  {RESULTS_DIR}/evaluation_results.json")
print(f"  {RESULTS_DIR}/zero_shot_vs_finetuned_v3_1.md")
print(f"  {RESULTS_DIR}/finetuned_test_predictions.jsonl")
print(f"  {RESULTS_DIR}/confusion_matrix_tme.png")
print(f"  {RESULTS_DIR}/calibration_curves.png")
print(f"  {RESULTS_DIR}/zero_shot_vs_finetuned.png")
print(f"{'=' * 60}")


PHASE 6 v3.1 — EVALUATION COMPLETE

  Model: google/medgemma-1.5-4b-it + LoRA v3.1 adapters
  Test samples: 94

  --- Key Metrics (Fine-Tuned v3.1) ---
  CD274 Bal. Acc:    0.5106  (target: >0.70)
  MSI Bal. Acc:      N/A  (target: >0.75)
    ⚠ Only 0 MSI-H samples — metric statistically unreliable
  TME Accuracy:      0.2747  (target: >0.65)
  TIL Spearman ρ:    0.1194  (target: >0.60)
  Immune Score MAE:  0.2071  (target: <0.15)
  JSON Parse Rate:   1.0

  --- Calibration ---
  Temperature: 10.0
  ECE before:  0.7894  (target: <0.10)
  ECE after:   0.524

  --- v2 Metrics (for reference) ---
  v2 CD274 Bal. Acc: 0.5638  → v3.1: 0.5106
  v2 MSI Bal. Acc:   0.5000  → v3.1: N/A  (⚠ 0 MSI-H patients in test)
  v2 TME Accuracy:   0.2473  → v3.1: 0.2747
  v2 TIL Spearman:   -0.1740 → v3.1: 0.1194
  v2 Immune MAE:     0.3032  → v3.1: 0.2071

  --- V3.1 Fixes Applied ---
  1. Response-only loss masking ✓
  2. No modules_to_save (weight tying preserved) ✓
  3. 3 epochs + early stopping (pati